In [14]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime

In [15]:
df = pd.read_csv('online_retail_II.csv', encoding='latin-1')

print(df.shape)
print(df.head())
print(df.dtypes)
print(df.isnull().sum())

(1067371, 8)
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

           InvoiceDate  Price  Customer ID         Country  
0  2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3  2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4  2009-12-01 07:45:00   1.25      13085.0  United Kingdom  
Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object
In

In [16]:
# Drop rows with missing Customer ID and Description
df.dropna(subset=['Customer ID', 'Description'], inplace=True)

# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Remove cancelled orders (Invoice starting with 'C')
df = df[~df['Invoice'].astype(str).str.startswith('C')]

# Remove rows where Quantity or Price is <= 0
df = df[df['Quantity'] > 0]
df = df[df['Price'] > 0]

# Convert Customer ID to integer
df['Customer ID'] = df['Customer ID'].astype(int)

# Confirm cleaned shape
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())

(805549, 8)
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID             int32
Country                object
dtype: object
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64


In [17]:
# Define reference date (1 day after last invoice date)
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

# Create Revenue column
df['Revenue'] = df['Quantity'] * df['Price']

# Calculate RFM metrics per customer
rfm = df.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('Revenue', 'sum')
).reset_index()

print(rfm.shape)
print(rfm.head(10))
print(rfm.describe())

(5878, 4)
   Customer ID  Recency  Frequency  Monetary
0        12346      326         12  77556.46
1        12347        2          8   5633.32
2        12348       75          5   2019.40
3        12349       19          4   4428.69
4        12350      310          1    334.40
5        12351      375          1    300.93
6        12352       36         10   2849.84
7        12353      204          2    406.76
8        12354      232          1   1079.40
9        12355      214          2    947.61
        Customer ID      Recency    Frequency       Monetary
count   5878.000000  5878.000000  5878.000000    5878.000000
mean   15315.313542   201.331916     6.289384    3018.616737
std     1715.572666   209.338707    13.009406   14737.731040
min    12346.000000     1.000000     1.000000       2.950000
25%    13833.250000    26.000000     1.000000     348.762500
50%    15314.500000    96.000000     3.000000     898.915000
75%    16797.750000   380.000000     7.000000    2307.090000
max    

In [18]:
# Assign scores 1-5 using quintiles
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=[5, 4, 3, 2, 1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=5, labels=[1, 2, 3, 4, 5])

# Combine into single RFM score
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

print(rfm.head(10))
print(rfm['RFM_Score'].value_counts().head(10))

   Customer ID  Recency  Frequency  Monetary R_Score F_Score M_Score RFM_Score
0        12346      326         12  77556.46       2       5       5       255
1        12347        2          8   5633.32       5       4       5       545
2        12348       75          5   2019.40       3       4       4       344
3        12349       19          4   4428.69       5       3       5       535
4        12350      310          1    334.40       2       1       2       212
5        12351      375          1    300.93       2       1       2       212
6        12352       36         10   2849.84       4       5       4       454
7        12353      204          2    406.76       2       2       2       222
8        12354      232          1   1079.40       2       1       3       213
9        12355      214          2    947.61       2       2       3       223
RFM_Score
555    474
111    320
455    249
121    173
344    167
211    162
112    149
444    147
122    137
544    134
Name: count

In [19]:
def assign_segment(row):
    r = int(row['R_Score'])
    f = int(row['F_Score'])
    m = int(row['M_Score'])

    if r >= 4 and f >= 4 and m >= 4:
        return 'Champion'
    elif r >= 3 and f >= 3 and m >= 3:
        return 'Loyal Customer'
    elif r >= 4 and f <= 2:
        return 'New Customer'
    elif r >= 3 and f >= 2 and m >= 2:
        return 'Potential Loyalist'
    elif r <= 2 and f >= 3 and m >= 3:
        return 'At Risk'
    elif r <= 2 and f >= 4 and m >= 4:
        return 'Cannot Lose Them'
    elif r == 1 and f == 1:
        return 'Lost'
    else:
        return 'Need Attention'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)

print(rfm['Segment'].value_counts())
print(rfm.head(10))

Segment
Need Attention        1517
Champion              1300
Loyal Customer        1134
At Risk                615
Lost                   513
New Customer           443
Potential Loyalist     356
Name: count, dtype: int64
   Customer ID  Recency  Frequency  Monetary R_Score F_Score M_Score  \
0        12346      326         12  77556.46       2       5       5   
1        12347        2          8   5633.32       5       4       5   
2        12348       75          5   2019.40       3       4       4   
3        12349       19          4   4428.69       5       3       5   
4        12350      310          1    334.40       2       1       2   
5        12351      375          1    300.93       2       1       2   
6        12352       36         10   2849.84       4       5       4   
7        12353      204          2    406.76       2       2       2   
8        12354      232          1   1079.40       2       1       3   
9        12355      214          2    947.61       2     

In [20]:
# Connect to SQLite database (creates file if not exists)
conn = sqlite3.connect('rfm_database.db')

# Load rfm dataframe into SQLite table
rfm.to_sql('rfm_segments', conn, if_exists='replace', index=False)

print("Data loaded successfully")
print(f"Total records loaded: {len(rfm)}")

# Verify it loaded
verify = pd.read_sql("SELECT * FROM rfm_segments LIMIT 5", conn)
print(verify)

Data loaded successfully
Total records loaded: 5878
   Customer ID  Recency  Frequency  Monetary R_Score F_Score M_Score  \
0        12346      326         12  77556.46       2       5       5   
1        12347        2          8   5633.32       5       4       5   
2        12348       75          5   2019.40       3       4       4   
3        12349       19          4   4428.69       5       3       5   
4        12350      310          1    334.40       2       1       2   

  RFM_Score         Segment  
0       255         At Risk  
1       545        Champion  
2       344  Loyal Customer  
3       535  Loyal Customer  
4       212  Need Attention  


In [21]:
# Query 1 - Segment distribution
q1 = pd.read_sql("""
    SELECT Segment, 
           COUNT(*) as Customer_Count,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM rfm_segments), 2) as Percentage
    FROM rfm_segments
    GROUP BY Segment
    ORDER BY Customer_Count DESC
""", conn)
print("=== Segment Distribution ===")
print(q1)

# Query 2 - Average RFM metrics per segment
q2 = pd.read_sql("""
    SELECT Segment,
           ROUND(AVG(Recency), 1) as Avg_Recency,
           ROUND(AVG(Frequency), 1) as Avg_Frequency,
           ROUND(AVG(Monetary), 2) as Avg_Monetary
    FROM rfm_segments
    GROUP BY Segment
    ORDER BY Avg_Monetary DESC
""", conn)
print("\n=== Average RFM Metrics per Segment ===")
print(q2)

# Query 3 - High value at risk customers
q3 = pd.read_sql("""
    SELECT `Customer ID`, Recency, Frequency, Monetary, RFM_Score
    FROM rfm_segments
    WHERE Segment = 'At Risk'
    AND Monetary > 2000
    ORDER BY Monetary DESC
    LIMIT 10
""", conn)
print("\n=== Top 10 High Value At Risk Customers ===")
print(q3)

# Query 4 - Revenue contribution by segment
q4 = pd.read_sql("""
    SELECT Segment,
           ROUND(SUM(Monetary), 2) as Total_Revenue,
           ROUND(SUM(Monetary) * 100.0 / (SELECT SUM(Monetary) FROM rfm_segments), 2) as Revenue_Percentage
    FROM rfm_segments
    GROUP BY Segment
    ORDER BY Total_Revenue DESC
""", conn)
print("\n=== Revenue Contribution by Segment ===")
print(q4)

=== Segment Distribution ===
              Segment  Customer_Count  Percentage
0      Need Attention            1517       25.81
1            Champion            1300       22.12
2      Loyal Customer            1134       19.29
3             At Risk             615       10.46
4                Lost             513        8.73
5        New Customer             443        7.54
6  Potential Loyalist             356        6.06

=== Average RFM Metrics per Segment ===
              Segment  Avg_Recency  Avg_Frequency  Avg_Monetary
0            Champion         20.0           17.1       9329.32
1             At Risk        359.5            5.6       2517.23
2      Loyal Customer         73.4            5.9       2295.58
3        New Customer         28.1            1.5        890.83
4  Potential Loyalist         79.8            2.7        602.15
5      Need Attention        343.3            1.6        446.84
6                Lost        568.2            1.0        345.37

=== Top 10 High V

In [23]:
import os

# Export RFM data to CSV for Power BI
rfm.to_csv('rfm_final.csv', index=False)

# Export SQL query results for Power BI
q1.to_csv('segment_distribution.csv', index=False)
q2.to_csv('segment_metrics.csv', index=False)
q4.to_csv('segment_revenue.csv', index=False)

# Close SQLite connection
conn.close()

print("All files exported successfully")
print(os.listdir())

All files exported successfully
['.ipynb_checkpoints', 'online_retail_II.csv', 'rfm_database.db', 'rfm_final.csv', 'rfm_pipeline.ipynb', 'segment_distribution.csv', 'segment_metrics.csv', 'segment_revenue.csv']
